# Automated Monitoring & Alerting Pipeline for Engineering Systems
## Karla Peña Ramírez

We want to determine the most frequent alarms triggered during the Commissioning Camera (ComCam, was a smaller and simplified version of the full 3.2-billion-pixel LSST Camera (LSSTCam) on-sky campaign. Here are presented histogram visualizations for the alarm names, reasons and acknowledgement status. It is also presented time series visualizations showing the number of alarms per day, and by hour and day of the week.

In [ ]:
#Setting packages
import asyncio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
import time
import warnings
import seaborn as sns
import datetime as dt
from scipy import stats
from matplotlib.dates import DateFormatter
from astropy.table import Table, join
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.ensemble import IsolationForest
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from lsst.utils.packages import getEnvironmentPackages
#%matplotlib widget

from astropy.time import Time, TimeDelta
from lsst_efd_client.efd_helper import merge_packed_time_series
from lsst.meas.algorithms.installGaussianPsf import FwhmPerSigma
from lsst.daf.butler import Butler 
#from tqdm.notebook import tqdm

# Ignore the many warning messages from ``merge_packed_time_series``
warnings.simplefilter(action="ignore", category=FutureWarning)

#-------Clients
from lsst_efd_client import EfdClient
client = EfdClient("usdf_efd")

#from lsst.summit.utils.efdUtils import makeEfdClient
#client = makeEfdClient()

#from lsst.summit.utils import ConsDbClient
#os.environ["no_proxy"] += ",.consdb"
#url="http://consdb-pq.consdb:8080/consdb"
#consdb=ConsDbClient(url)
#-------Clients

def print_session_info():
    # Time info
    print(f"# Session Info on {time.strftime('%Y-%m-%d at %H:%M:%S %Z', time.localtime(time.time()))}\n")

    # Python info
    print(f"## Python Interpreter\n\nVersion: {sys.version}  \nExecutable: {sys.executable}\n")

    # LSST info
    packages = getEnvironmentPackages(True)
    dev_packages = {"lsst_distrib": packages["lsst_distrib"]}
    dev_packages.update({k: v.split("@")[0] for k, v in packages.items() if "LOCAL" in v})
    print("## Science Pipelines\n\n" + "\n".join(f"{k:<20} {v}" for k, v in dev_packages.items()))

Retrieving the alarms during ComCam campaign

In [ ]:
#ComCam campaign window
start = Time("2024-10-17 00:00:00Z", scale="utc")
end = Time("2024-12-12 00:00:00Z", scale="utc")

In [ ]:
#Data retrieval 1. Watcher.
query = f"""
SELECT 
    "name" AS "Name", 
    "reason" AS "Reason", 
    "acknowledged" AS "Acknowledged",
    "severity" AS "Severity",
    "mutedBy" AS "Muted"
FROM "lsst.sal.Watcher.logevent_alarm" 
WHERE time > '{start.isot}Z' 
AND time < '{end.isot}Z'
"""

watcher = await client.influx_client.query(query)

print(f"Total number of alarms: {len(watcher)}")
print(f"Number of unique alarm reasons: {watcher['Reason'].nunique()}")
print("\nAcknowledgment rate: {:.2f}%".format(
    (watcher['Acknowledged'].sum() / len(watcher)) * 100
))

In [ ]:
#Data exploration
#Alarm Distribution by Acknowledgment Status
plt.figure(figsize=(18, 3))
sns.countplot(data=watcher, x='Acknowledged')
plt.title('Distribution of Acknowledged vs Unacknowledged Alarms')
plt.show()


#Alarm Distribution by Severity Status
plt.figure(figsize=(18, 3))
sns.countplot(data=watcher, x='Severity')
plt.title('Distribution of Severity Flag')
plt.show()


From https://github.com/lsst-ts/ts_xml/blob/develop/python/lsst/ts/xml/enums/Watcher.py we know that the severity e is defined as:

NONE = 1\
WARNING = 2\
SERIOUS = 3\
CRITICAL = 4

So let's discard the alarms with severity=NONE and those muted by some user.

In [ ]:
# Filter for severity 2,3,4 only and muted severity 'None'
watcher_filtered = watcher[
    (watcher['Severity'].isin([2, 3, 4])) & 
    (watcher['Muted'] == '')]

print(f"Total number of alarms: {len(watcher_filtered)}")
print(f"Number of unique alarm reasons: {watcher_filtered['Reason'].nunique()}")
print("\nAcknowledgment rate: {:.2f}%".format(
    (watcher['Acknowledged'].sum() / len(watcher_filtered)) * 100
))

In [ ]:
#Visualizations
#Truncate long strings
def truncate_string(s, max_length=50):
    return s if len(s) <= max_length else s[:max_length-3] + '...'

#Name of the Top 20 Most Frequent Alarms (Not muted)
plt.figure(figsize=(15, 10))
name_counts = watcher_filtered['Name'].apply(truncate_string).value_counts().head(20)
sns.barplot(x=name_counts.values, y=name_counts.index)
plt.title('Name of the Top 20 Most Frequent Alarms (not muted)')
plt.xlabel('Count')
plt.ylabel('Name')
plt.tight_layout()
plt.margins(y=0.02)
plt.show()

#Top 20 Most Frequent Reasons for Alarms (Not muted)
plt.figure(figsize=(15, 10))
reason_counts = watcher_filtered['Reason'].apply(truncate_string).value_counts().head(20)
sns.barplot(x=reason_counts.values, y=reason_counts.index)
plt.title('Top 20 Most Frequent Alarm Reasons (not muted)')
plt.xlabel('Count')
plt.ylabel('Reason')
plt.tight_layout()
plt.margins(y=0.02)
plt.show()

#Top 10 Alarm Names with Acknowledgment Status
top_10_names = watcher_filtered['Name'].apply(truncate_string).value_counts().head(10).index
plt.figure(figsize=(15, 8))
name_ack = watcher_filtered[watcher_filtered['Name'].isin(top_10_names)]
sns.countplot(data=name_ack, y='Name', hue='Acknowledged')
plt.title('Top 10 Alarm Names by Acknowledgment Status (not muted)')
plt.tight_layout()
plt.show()


#Top 10 Alarm Names with Severity Status
top_10_names = watcher_filtered['Name'].apply(truncate_string).value_counts().head(10).index
plt.figure(figsize=(15, 8))
name_sev = watcher_filtered[watcher_filtered['Name'].isin(top_10_names)]
sns.countplot(data=name_sev, y='Name', hue='Severity')
plt.title('Top 10 Alarm Names by Severity Status (not muted)')
plt.tight_layout()
plt.show()


#####Time Series Analysis
#Alarms per Date
watcher_filtered = watcher[
    (watcher['Severity'].isin([2, 3, 4])) & 
    (watcher['Muted'] == '')
].copy()
watcher_filtered['date'] = pd.to_datetime(watcher_filtered.index).date
daily_alarms = watcher_filtered.groupby('date').size()
plt.figure(figsize=(15, 5))
daily_alarms.plot(kind='line')
plt.title('Number of Alarms per Day (not muted)')
plt.xlabel('Date')
plt.ylabel('Number of Alarms')
plt.grid(True)  
plt.xticks(rotation=45)  
plt.tight_layout()
plt.show()


#Alarms by Hour and Day of Week
watcher_filtered['hour'] = pd.to_datetime(watcher_filtered.index).hour
watcher_filtered['day_of_week'] = pd.to_datetime(watcher_filtered.index).day_name()
watcher_filtered['date'] = pd.to_datetime(watcher_filtered.index).date
#Moving from counts to means per day per hour
daily_counts = watcher_filtered.groupby(['date', 'day_of_week', 'hour']).size().reset_index(name='count')
alarm_heatmap = daily_counts.groupby(['day_of_week', 'hour'])['count'].mean().unstack()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
alarm_heatmap = alarm_heatmap.reindex(day_order)
plt.figure(figsize=(18, 8))
sns.heatmap(alarm_heatmap, cmap='YlOrRd', annot=True, fmt='.0f')
plt.title('Average Number of Alarms by Hour and Day of Week (not muted)')
plt.tight_layout()
plt.show()

Identifying the ComCam on sky time period per night.

In [ ]:
#Identify the repo where all the data lives and the latest consolidated processing collection
#Run > butler query-collections /repo/main "*DRP*w_2025_0*" | grep CHAINED
repo = '/repo/main'
instrument = "LSSTComCam"
collection = 'LSSTComCam/runs/DRP/DP1/w_2025_08/DM-49029'
butler = Butler(repo, collections=collection)
registry = butler.registry

In [ ]:
#Query the metadata for the **`exposure`** dimension, limiting the results to this particular instrument and range of days of observation:
instrument = 'LSSTComCam'
day_obs_start = 20241017
day_obs_end = 20241212
query="instrument='%s' AND day_obs>=%d AND day_obs<=%d" % (instrument, day_obs_start, day_obs_end)
results = registry.queryDimensionRecords('exposure', where=query)

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Stop executing if there are no results returned:
n_results = results.count()

if n_results <= 0:
    raise StopExecution
else:
    print("""There are %d results returned from querying the butler for instrument %s between dates %d and %d (inclusive).""" % 
          (n_results, instrument, day_obs_start, day_obs_end))

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Instantiate a pandas `DataFrame` with useful columns available in the `exposure` dimension:
df_exp = pd.DataFrame(columns=['id', 'obs_id','day_obs', 'seq_num',
                                    'time_start','time_end' ,'type', 'reason', 
                                    'target','filter','zenith_angle',
                                    'expos','ra','dec','skyangle',
                                    'azimuth','zenith','science_program',
                                    'jd','mjd'])

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Read the query results into the new pandas `DataFrame`:
for count, info in enumerate(results):
    
    try:

        df_exp.loc[count] = [info.id, info.obs_id, info.day_obs, info.seq_num, 
                                  info.timespan.begin.utc.iso,
                                  info.timespan.end.utc.iso, 
                                  info.observation_type, info.observation_reason, info.target_name, 
                                  info.physical_filter, info.zenith_angle, 
                                  info.exposure_time,info.tracking_ra, info.tracking_dec, 
                                  info.sky_angle,info.azimuth ,info.zenith_angle, 
                                  info.science_program, info.timespan.begin.jd, info.timespan.begin.mjd]

    except:
    
        print(">>>   Unexpected error:", sys.exc_info()[0])
        info_timespan_begin_to_string = "2021-01-01 00:00:00.00"
        info_timespan_end_to_string = "2051-01-01 00:00:00.00"
        info_timespan_begin_jd = 0
        info_timespan_begin_mjd = 0
        df_exp.loc[count] = [info.id, info.obs_id, info.day_obs, info.seq_num, 
                                  pd.to_datetime(info_timespan_begin_to_string),
                                  pd.to_datetime(info_timespan_end_to_string), 
                                  info.observation_type, info.observation_reason, info.target_name, 
                                  info.physical_filter, info.zenith_angle, 
                                  info.exposure_time,info.tracking_ra, info.tracking_dec, 
                                  info.sky_angle,info.azimuth ,info.zenith_angle, 
                                  info.science_program, info_timespan_begin_jd, info_timespan_begin_mjd ]  

In [ ]:
#Taken from vv-team-notebooks/reports/TargetReport.ipynb
#Clean-up the dataframe
#Re-cast the `id`, `day_obs`, and `seq_num` rows as `int`'s:
df_exp = df_exp.astype({"id": int,'day_obs': int,'seq_num':int})
# Replace `NaN`'s in the `ra` and `dec` columns with zero.  
df_exp['ra'] = df_exp['ra'].fillna(0)
df_exp['dec'] = df_exp['dec'].fillna(0)
#Select only on-sky data
df_exp.type.unique()
df_open = df_exp[(df_exp.type == 'science') | (df_exp.type == 'cwfs') | (df_exp.type == 'focus') | (df_exp.type == 'acq')| (df_exp.type == 'flat')]

In [ ]:
#Group data by dayobs identifying the extremme dates for the exposures.
result_on_sky = df_open.groupby('day_obs').agg({
    'time_start': 'min',
    'time_end': 'max'
}).reset_index()

In [ ]:
def read_time_data():
    df = result_on_sky
    df['time_start'] = pd.to_datetime(df['time_start'])
    df['time_end'] = pd.to_datetime(df['time_end'])
    return df
    
def get_time_range(df, day_obs):
    day_data = df[df['day_obs'] == day_obs].iloc[0]
    return day_data['time_start'], day_data['time_end']

In [ ]:
#Assited by claude 3.5 sonnet
#Watcher
async def get_watcher_data(client, day_obs):
    """Get watcher data for a specific day_obs."""
    df = read_time_data()
    start_time, end_time = get_time_range(df, int(day_obs))
        
    query = f"""
    SELECT 
        "name" AS "Name", 
        "reason" AS "Reason", 
        "acknowledged" AS "Acknowledged"
    FROM "lsst.sal.Watcher.logevent_alarm" 
    WHERE time > '{start_time.isoformat()}Z' 
    AND time < '{end_time.isoformat()}Z' 
    """
    results = await client.influx_client.query(query)
    return pd.DataFrame(results)


async def get_all_watcher_data(client):
    """Get watcher data for all days."""
    df = read_time_data()
    all_data = []

    for _, row in df.iterrows():
        start_time = row['time_start']
        end_time = row['time_end']
        
        query = f"""
        SELECT 
            "name" AS "Name", 
            "reason" AS "Reason", 
            "acknowledged" AS "Acknowledged"
        FROM "lsst.sal.Watcher.logevent_alarm" 
        WHERE time > '{start_time.isoformat()}Z' 
        AND time < '{end_time.isoformat()}Z' 
        """
        results = await client.influx_client.query(query)
        all_data.append(pd.DataFrame(results))
        
    return pd.concat(all_data) if all_data else pd.DataFrame()

In [ ]:
# All period data:
watcher = await get_all_watcher_data(client)
print(f"Total number of alarms: {len(watcher)}")
print(f"Number of unique alarm reasons: {watcher['Reason'].nunique()}")
print("\nAcknowledgment rate: {:.2f}%".format(
    (watcher['Acknowledged'].sum() / len(watcher)) * 100
))

Producing the relevant data visualizations for this project:

In [ ]:
#Visualizations
#Truncate long strings
def truncate_string(s, max_length=50):
    return s if len(s) <= max_length else s[:max_length-3] + '...'

#Name of the Top 20 Most Frequent Alarms
plt.figure(figsize=(15, 10))
name_counts = watcher['Name'].apply(truncate_string).value_counts().head(20)
sns.barplot(x=name_counts.values, y=name_counts.index)
plt.title('Name of the Top 20 Most Frequent Alarms *on-sky*')
plt.xlabel('Count')
plt.ylabel('Name')
plt.tight_layout()
plt.margins(y=0.02)
plt.show()

#Top 20 Most Frequent Reasons for Alarms
plt.figure(figsize=(15, 10))
reason_counts = watcher['Reason'].apply(truncate_string).value_counts().head(20)
sns.barplot(x=reason_counts.values, y=reason_counts.index)
plt.title('Top 20 Most Frequent Alarm Reasons *on-sky*')
plt.xlabel('Count')
plt.ylabel('Reason')
plt.tight_layout()
plt.margins(y=0.02)
plt.show()

#Alarm Distribution by Acknowledgment Status
plt.figure(figsize=(18, 3))
sns.countplot(data=watcher, x='Acknowledged')
plt.title('Distribution of Acknowledged vs Unacknowledged Alarms *on-sky*')
plt.show()

#Top 10 Names with Acknowledgment Status
top_10_names = watcher['Name'].apply(truncate_string).value_counts().head(10).index
plt.figure(figsize=(15, 8))
name_ack = watcher[watcher['Name'].isin(top_10_names)]
sns.countplot(data=name_ack, y='Name', hue='Acknowledged')
plt.title('Top 10 Alarm Names by Acknowledgment Status *on-sky*')
plt.tight_layout()
plt.show()

#Top 10 Reasons with Acknowledgment Status
top_10_reasons = watcher['Reason'].apply(truncate_string).value_counts().head(10).index
plt.figure(figsize=(15, 8))
reason_ack = watcher[watcher['Reason'].isin(top_10_reasons)]
sns.countplot(data=reason_ack, y='Reason', hue='Acknowledged')
plt.title('Top 10 Alarm Reasons by Acknowledgment Status *on-sky*')
plt.tight_layout()
plt.show()



#####Time Series Analysis
#Alarms per Date
watcher['date'] = pd.to_datetime(watcher.index).date
daily_alarms = watcher.groupby('date').size()
plt.figure(figsize=(15, 5))
daily_alarms.plot(kind='line')
plt.title('Number of Alarms per Day *on-sky*')
plt.xlabel('Date')
plt.ylabel('Number of Alarms')
plt.tight_layout()
plt.show()


#Alarms by Hour and Day of Week
watcher['hour'] = pd.to_datetime(watcher.index).hour
watcher['day_of_week'] = pd.to_datetime(watcher.index).day_name()
watcher['date'] = pd.to_datetime(watcher.index).date
#Moving from counts to means per day per hour
daily_counts = watcher.groupby(['date', 'day_of_week', 'hour']).size().reset_index(name='count')
alarm_heatmap = daily_counts.groupby(['day_of_week', 'hour'])['count'].mean().unstack()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
alarm_heatmap = alarm_heatmap.reindex(day_order)
plt.figure(figsize=(18, 8))
sns.heatmap(alarm_heatmap, cmap='YlOrRd', annot=True, fmt='.0f')
plt.title('Average Number of Alarms by Hour and Day of Week')
plt.tight_layout()
plt.show()



In [ ]:
#Specific day period data:
watcher = await get_watcher_data(client, '20241115')
print(f"Total number of alarms: {len(watcher)}")
print(f"Number of unique alarm reasons: {watcher['Reason'].nunique()}")
print("\nAcknowledgment rate: {:.2f}%".format(
    (watcher['Acknowledged'].sum() / len(watcher)) * 100
))

In [ ]:
#Visualizations
#Truncate long strings
def truncate_string(s, max_length=50):
    return s if len(s) <= max_length else s[:max_length-3] + '...'

#Name of the Top 20 Most Frequent Alarms
plt.figure(figsize=(15, 10))
name_counts = watcher['Name'].apply(truncate_string).value_counts().head(20)
sns.barplot(x=name_counts.values, y=name_counts.index)
plt.title('Name of the Top 20 Most Frequent Alarms *on-sky*')
plt.xlabel('Count')
plt.ylabel('Name')
plt.tight_layout()
plt.margins(y=0.02)
plt.show()

#Top 20 Most Frequent Reasons for Alarms
plt.figure(figsize=(15, 10))
reason_counts = watcher['Reason'].apply(truncate_string).value_counts().head(20)
sns.barplot(x=reason_counts.values, y=reason_counts.index)
plt.title('Top 20 Most Frequent Alarm Reasons *on-sky*')
plt.xlabel('Count')
plt.ylabel('Reason')
plt.tight_layout()
plt.margins(y=0.02)
plt.show()

#Alarm Distribution by Acknowledgment Status
plt.figure(figsize=(18, 3))
sns.countplot(data=watcher, x='Acknowledged')
plt.title('Distribution of Acknowledged vs Unacknowledged Alarms *on-sky*')
plt.show()

#Top 10 Names with Acknowledgment Status
top_10_names = watcher['Name'].apply(truncate_string).value_counts().head(10).index
plt.figure(figsize=(15, 8))
name_ack = watcher[watcher['Name'].isin(top_10_names)]
sns.countplot(data=name_ack, y='Name', hue='Acknowledged')
plt.title('Top 10 Alarm Names by Acknowledgment Status *on-sky*')
plt.tight_layout()
plt.show()

#Top 10 Reasons with Acknowledgment Status
top_10_reasons = watcher['Reason'].apply(truncate_string).value_counts().head(10).index
plt.figure(figsize=(15, 8))
reason_ack = watcher[watcher['Reason'].isin(top_10_reasons)]
sns.countplot(data=reason_ack, y='Reason', hue='Acknowledged')
plt.title('Top 10 Alarm Reasons by Acknowledgment Status *on-sky*')
plt.tight_layout()
plt.show()


In [ ]:
#Reproducibility
print_session_info()